In [ ]:
import os
import re
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from underthesea import word_tokenize

print("🚀 BẮT ĐẦU KHỞI ĐỘNG HỆ THỐNG DEMO LOCAL (PHOBERT + GRU)...")

# ==========================================
# 1. NẠP TỪ ĐIỂN TEENCODE (Đường dẫn tương đối)
# ==========================================
dict_path = 'acronyms_dictionary.txt' 
ACRONYM_DICT = {}

print("🔍 Đang nạp từ điển teencode...")
try:
    with open(dict_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip() 
            if not line or line.startswith('#'):
                continue
            parts = line.split('=')
            if len(parts) >= 2:
                key = parts[0].strip().lower()
                value = parts[1].strip().lower()
                ACRONYM_DICT[key] = value
                
    print(f"✅ Đã nạp thành công {len(ACRONYM_DICT)} từ lóng/viết tắt từ máy tính!")
except Exception as e:
    print(f"⚠️ Lỗi đọc file teencode: {e}")

# ==========================================
# 2. HÀM TIỀN XỬ LÝ DỮ LIỆU CHUẨN ĐỒ ÁN
# ==========================================
def preprocess_text(text):
    text = text.lower()
    words = text.split()
    text = " ".join([ACRONYM_DICT.get(w, w) for w in words])
    text = re.sub(r'[^\s\wáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệóòỏõọôốồổỗộơớờởỡợíìỉĩịúùủũụưứừửữựýỳỷỹỵđ_]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = word_tokenize(text, format="text")
    return text

# ==========================================
# 3. ĐỊNH NGHĨA KIẾN TRÚC MÔ HÌNH (Khung xương)
# ==========================================
class PhoBERT_GRU(nn.Module):
    def __init__(self, num_classes):
        super(PhoBERT_GRU, self).__init__()
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base")
        
        self.gru = nn.GRU(input_size=self.phobert.config.hidden_size, 
                          hidden_size=256, 
                          num_layers=1, 
                          batch_first=True, 
                          bidirectional=True)
        
        self.fc = nn.Linear(256 * 2, num_classes) 
        
    def forward(self, input_ids, attention_mask):
        features = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = features.last_hidden_state 
        gru_out, _ = self.gru(sequence_output)
        
        # Lấy trạng thái đầu ra cuối cùng của GRU để phân loại
        last_hidden_state = gru_out[:, -1, :] 
        output = self.fc(last_hidden_state)
        return output

# ==========================================
# 4. NẠP TOKENIZER VÀ TRỌNG SỐ (.PTH)
# ==========================================
print("🔍 Đang nạp Tokenizer và Trọng số từ thư mục models_PhoBERT_GRU...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Đường dẫn tương đối lùi 1 cấp ra ngoài Notebooks rồi vào models_PhoBERT_GRU
base_path = '../models_PhoBERT_GRU'

senti_model_path = os.path.join(base_path, 'sentiment_model/best_phobert_gru_model.pth')
senti_tok_path = os.path.join(base_path, 'sentiment_model/best_phobert_gru_model_tokenizer')

topic_model_path = os.path.join(base_path, 'topic_model/best_phobert_gru_model.pth')
topic_tok_path = os.path.join(base_path, 'topic_model/best_phobert_gru_model_tokenizer')

if not os.path.exists(senti_model_path) or not os.path.exists(topic_model_path):
    print("❌ Lỗi: Không tìm thấy file trọng số .pth tương đối! Kiểm tra lại vị trí đặt notebook.")
else:
    # Nạp Tokenizer
    tokenizer_sentiment = AutoTokenizer.from_pretrained(senti_tok_path, local_files_only=True)
    tokenizer_topic = AutoTokenizer.from_pretrained(topic_tok_path, local_files_only=True)
    
    # Khởi tạo cấu trúc và nạp trọng số cảm xúc (3 nhãn)
    model_sentiment = PhoBERT_GRU(num_classes=3).to(device)
    model_sentiment.load_state_dict(torch.load(senti_model_path, map_location=device))
    model_sentiment.eval()
    
    # Khởi tạo cấu trúc và nạp trọng số chủ đề (4 nhãn)
    model_topic = PhoBERT_GRU(num_classes=4).to(device)
    model_topic.load_state_dict(topic_model_path, map_location=device) if hasattr(model_topic, 'load_state_dict') else None
    model_topic.load_state_dict(torch.load(topic_model_path, map_location=device))
    model_topic.eval()
    
    print(f"✅ Hệ thống PhoBERT + GRU Local đã sẵn sàng trên Card: {device}")

dict_sentiment = {0: '🔴 Tiêu cực', 1: '⚪ Trung tính', 2: '🟢 Tích cực'}
dict_topic = {0: '👨‍🏫 Giảng viên', 1: '📚 Chương trình', 2: '🏫 Cơ sở vật chất', 3: '❓ Khác'}

# ==========================================
# 5. VÒNG LẶP KIỂM THỬ
# ==========================================
print("\n" + "="*50)
print("🎯 CHƯƠNG TRÌNH DEMO PHOBERT + GRU (LOCAL VERSION)")
print("Nhập 'q' để thoát.")
print("="*50 + "\n")

while True:
    raw_text = input("✍️ Mời nhập câu nhận xét: ")
    if raw_text.lower() in ['q', 'exit', 'thoat']:
        print("👋 Đã thoát chương trình!")
        break
    if not raw_text.strip():
        continue

    clean_text = preprocess_text(raw_text)
    inputs = tokenizer_sentiment(clean_text, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
    
    with torch.no_grad():
        input_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']
        
        out_s = model_sentiment(input_ids=input_ids, attention_mask=attention_mask)
        out_t = model_topic(input_ids=input_ids, attention_mask=attention_mask)
        
        pred_s_idx = torch.argmax(out_s, dim=1).item()
        pred_t_idx = torch.argmax(out_t, dim=1).item()
        
    print("-" * 50)
    print(f"Câu gốc        : {raw_text}")
    print(f"Câu sau xử lý  : {clean_text}")
    print(f"Chủ đề         : {dict_topic[pred_t_idx]}")
    print(f"Cảm xúc        : {dict_sentiment[pred_s_idx]}")
    print("-" * 50 + "\n")